# Romleg analyse av beredskapsressursar i Kristiansand
## Totalforsvaret 2025–2026

Denne notebooken utfører romleg (spatial) analyse av beredskapsdata i Kristiansand-regionen.
Tematikken er knytt til **Totalforsvaret** — vi analyserer distribusjonen av hjertestartarar (AED),
brannstasjonar, sjukehus og andre beredskapsressursar, og brukar vektoranalyse og rasteranalyse
for å vurdere beredskapsdekninga.

### Verktøy som vert brukt
- **Pandas** — databehandling og aggregering
- **GeoPandas** — romleg databehandling, buffer, overlay
- **DuckDB** — SQL-spørjingar mot romleg data
- **Folium** — interaktiv kartvisualisering
- **Matplotlib** — statisk visualisering
- **GDAL** — rasteranalyse (DEM, slope, hillshade)
- **Rasterio** — lesing av rasterdata

### Datasett
1. **AED-hjertestartarar** — frå Hjertestarterregister API (JSON)
2. **Beredskapsressursar** — lokal GeoJSON-fil (sjukehus, legevakt, politi m.m.)
3. **Kommunegrenser** — frå GeoNorge WFS (administrerte grenser)
4. **Høgdedata (DEM)** — frå Kartverket / Copernicus for rasteranalyse

## 1. Oppsett og import av bibliotek

Vi importerer alle nødvendige bibliotek for romleg analyse. GeoPandas bygger på Pandas
og gir oss verktøy for geografisk databehandling med Shapely-geometriar.

In [ ]:
# Installer eventuelle manglande pakkar
# %pip install geopandas folium matplotlib duckdb rasterio requests shapely pyproj

import pandas as pd
import geopandas as gpd
import folium
from folium.plugins import MarkerCluster
import matplotlib.pyplot as plt
import duckdb
import json
import requests
import os
import numpy as np
from shapely.geometry import Point, Polygon, MultiPolygon
from shapely.ops import unary_union
import warnings
warnings.filterwarnings('ignore')

print(f"Pandas versjon: {pd.__version__}")
print(f"GeoPandas versjon: {gpd.__version__}")
print("Alle bibliotek er lasta inn.")

## 2. Data Ingest — Henting av datasett

### 2.1 Datasett 1: Beredskapsressursar (lokal GeoJSON)

Vi les inn den lokale GeoJSON-fila som inneheld kritiske beredskapsressursar i Kristiansand:
sjukehus, legevakt, AMK, politi, sivilforsvar, Røde Kors og liknande.
Denne fila er kuratert frå OpenStreetMap og offentlege kjelder.

In [ ]:
# Les inn lokal GeoJSON med beredskapsressursar
beredskap_gdf = gpd.read_file('app/data/norwegian_landmarks.geojson')

print(f"Antal beredskapsressursar: {len(beredskap_gdf)}")
print(f"Kolonnar: {list(beredskap_gdf.columns)}")
print(f"CRS: {beredskap_gdf.crs}")
print(f"Geometritypar: {beredskap_gdf.geometry.geom_type.value_counts().to_dict()}")
print()
beredskap_gdf[['name', 'category', 'description']].head(11)

### 2.2 Datasett 2: AED-hjertestartarar frå Hjertestarterregister API

Vi hentar hjertestartardata frå det nasjonale Hjertestarterregisteret sitt REST API.
API-et krev OAuth 2.0-autentisering. Vi søkjer i ein radius rundt Kristiansand sentrum
og konverterer resultatet til ein GeoDataFrame.

Dersom API-et ikkje er tilgjengeleg, lastar vi ein fallback-fil.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

def fetch_aed_data():
    """Hent AED-data frå Hjertestarterregister API med OAuth 2.0."""
    client_id = os.getenv('HJERTESTARTERREGISTER_CLIENT_ID', '')
    client_secret = os.getenv('HJERTESTARTERREGISTER_CLIENT_SECRET', '')
    
    if not client_id or not client_secret:
        print("⚠ API-nøklar ikkje tilgjengelege — brukar fallback")
        return None
    
    # OAuth 2.0 token
    token_url = 'https://www.hjertestarterregisteret.no/api/oauth/token'
    token_resp = requests.post(token_url, data={
        'grant_type': 'client_credentials',
        'client_id': client_id,
        'client_secret': client_secret
    }, timeout=15)
    
    if token_resp.status_code != 200:
        print(f"⚠ OAuth feil: {token_resp.status_code}")
        return None
    
    token = token_resp.json().get('access_token')
    
    # Søk etter AED-ar i Kristiansand
    search_url = 'https://www.hjertestarterregisteret.no/api/asset/search'
    search_resp = requests.get(search_url, params={
        'latitude': 58.1414,
        'longitude': 8.0842,
        'distance': 15000,
        'maxRows': 5000
    }, headers={'Authorization': f'Bearer {token}'}, timeout=20)
    
    if search_resp.status_code == 200:
        return search_resp.json()
    print(f"⚠ Søk feil: {search_resp.status_code}")
    return None

# Hent data
raw_aed = fetch_aed_data()

if raw_aed and 'ASSETS' in raw_aed:
    assets = raw_aed['ASSETS']
    aed_records = []
    for a in assets:
        lat = a.get('SITE_LATITUDE')
        lng = a.get('SITE_LONGITUDE')
        if lat and lng:
            aed_records.append({
                'asset_id': a.get('ASSET_ID'),
                'name': a.get('SITE_NAME', 'Ukjend'),
                'address': a.get('SITE_ADDRESS', ''),
                'post_code': a.get('SITE_POST_CODE', ''),
                'post_area': a.get('SITE_POST_AREA', ''),
                'is_open': a.get('IS_OPEN', 'N') == 'Y',
                'opening_hours': a.get('OPENING_HOURS_TEXT', ''),
                'latitude': float(lat),
                'longitude': float(lng),
            })
    aed_df = pd.DataFrame(aed_records)
    geometry = [Point(row.longitude, row.latitude) for _, row in aed_df.iterrows()]
    aed_gdf = gpd.GeoDataFrame(aed_df, geometry=geometry, crs='EPSG:4326')
    print(f"✓ Henta {len(aed_gdf)} AED-ar frå Hjertestarterregister API")
else:
    # Fallback: generer demodata basert på prosjektets kjende data
    print("⚠ Brukar genererte demodata for AED-ar (API ikkje tilgjengeleg)")
    np.random.seed(42)
    n = 263
    lats = 58.1414 + np.random.normal(0, 0.04, n)
    lngs = 8.0842 + np.random.normal(0, 0.06, n)
    aed_df = pd.DataFrame({
        'asset_id': range(1, n+1),
        'name': [f'AED {i}' for i in range(1, n+1)],
        'address': [f'Adresse {i}, Kristiansand' for i in range(1, n+1)],
        'post_code': np.random.choice(['4608', '4610', '4611', '4612', '4614', '4615', '4616', '4617', '4620', '4621', '4622', '4623', '4624', '4625', '4626', '4628', '4630', '4631', '4632', '4633'], n),
        'post_area': 'Kristiansand',
        'is_open': np.random.choice([True, False], n, p=[0.75, 0.25]),
        'opening_hours': np.random.choice(['Alltid', 'Dagtid', '08:00-16:00', '07:00-22:00'], n),
        'latitude': lats,
        'longitude': lngs
    })
    geometry = [Point(row.longitude, row.latitude) for _, row in aed_df.iterrows()]
    aed_gdf = gpd.GeoDataFrame(aed_df, geometry=geometry, crs='EPSG:4326')

print(f"\nAED DataFrame info:")
print(f"  Rader: {len(aed_gdf)}")
print(f"  Kolonnar: {list(aed_gdf.columns)}")
print(f"  CRS: {aed_gdf.crs}")
print(f"  Åpne: {aed_gdf['is_open'].sum()}, Stengt: {(~aed_gdf['is_open']).sum()}")
aed_gdf.head()

### 2.3 Datasett 3: Kommunegrenser frå GeoNorge WFS

Vi hentar kommunegrenser frå GeoNorge sin WFS-teneste (OGC Web Feature Service).
Dette gir oss polygondata for Kristiansand kommune (nr. 4204) som vi brukar til
romleg filtrering og aggregering seinare.

In [ ]:
# Hent kommunegrenser frå GeoNorge WFS
wfs_url = 'https://wfs.geonorge.no/skwms1/wfs.administrative_enheter'
params = {
    'service': 'WFS',
    'version': '2.0.0',
    'request': 'GetFeature',
    'typeNames': 'Kommune',
    'outputFormat': 'application/json',
    'srsName': 'EPSG:4326',
    'CQL_FILTER': "kommunenummer='4204'",  # Kristiansand
    'count': '10'
}

try:
    resp = requests.get(wfs_url, params=params, timeout=30)
    resp.raise_for_status()
    kommune_gdf = gpd.GeoDataFrame.from_features(resp.json()['features'], crs='EPSG:4326')
    print(f"✓ Henta {len(kommune_gdf)} kommunegrense(r) frå GeoNorge WFS")
    print(f"  Kolumnar: {list(kommune_gdf.columns)}")
    kommune_gdf.head()
except Exception as e:
    print(f"⚠ Kommunegrenser feil: {e}")
    print("Brukar ein enkel bounding box for Kristiansand i staden.")
    from shapely.geometry import box
    kristiansand_bbox = box(7.85, 58.05, 8.30, 58.25)
    kommune_gdf = gpd.GeoDataFrame(
        [{'kommunenummer': '4204', 'kommunenavn': 'Kristiansand', 'geometry': kristiansand_bbox}],
        crs='EPSG:4326'
    )
    print(f"✓ Oppretta fallback kommunegrense (bounding box)")

print(f"\nCRS: {kommune_gdf.crs}")
print(f"Areal (EPSG:4326 grader²): {kommune_gdf.geometry.area.values}")

## 3. Filtrering og visualisering

### 3.1 Attributtfiltrering: Åpne vs. stengte AED-ar

Vi filtrerer AED-datsettet på attributtet `is_open` for å skilje mellom hjertestartarar
som er tilgjengelege for publikum og dei som er stengte. Resultatet viser kor mange
som er tilgjengelege for bruk i ein nødsituasjon — kritisk for beredskapsvurdering.

In [ ]:
# Attributtfiltrering — del opp i åpne og stengte
aed_open = aed_gdf[aed_gdf['is_open'] == True].copy()
aed_closed = aed_gdf[aed_gdf['is_open'] == False].copy()

print(f"Totalt AED-ar:  {len(aed_gdf)}")
print(f"  Åpne:         {len(aed_open)} ({len(aed_open)/len(aed_gdf)*100:.1f}%)")
print(f"  Stengte:      {len(aed_closed)} ({len(aed_closed)/len(aed_gdf)*100:.1f}%)")

# Visualisering med geopandas.plot()
fig, ax = plt.subplots(1, 1, figsize=(12, 8))

# Vis kommunen som bakgrunn
kommune_gdf.plot(ax=ax, facecolor='#f0f0f0', edgecolor='#333', linewidth=1.5, alpha=0.5)

# AED-ar fargekoda etter status
aed_closed.plot(ax=ax, color='#e94560', markersize=20, alpha=0.6, label=f'Stengt ({len(aed_closed)})')
aed_open.plot(ax=ax, color='#27ae60', markersize=20, alpha=0.7, label=f'Åpen ({len(aed_open)})')

ax.set_title('Hjertestartarar (AED) i Kristiansand — Åpne vs. Stengte', fontsize=14, fontweight='bold')
ax.set_xlabel('Lengdegrad')
ax.set_ylabel('Breddgrad')
ax.legend(loc='upper right', fontsize=10)
plt.tight_layout()
plt.show()

print("\nKartet viser at dei fleste AED-ane er tilgjengelege (grøne).")
print("Beredskapsmessig er det viktig at flest mogleg er åpne for rask tilgang.")

### 3.2 Romleg filtrering: AED-ar innanfor Kristiansand kommune

Vi brukar romleg filtrering (`sjoin`) for å finne AED-ar som ligg innanfor
Kristiansand kommune sine grenser. Dette er relevant for å vurdere om
beredskapsdekninga er god nok innanfor kommunen.

In [ ]:
# Romleg filtrering — AED-ar innanfor kommunegrensa
aed_in_kommune = gpd.sjoin(aed_gdf, kommune_gdf, how='inner', predicate='within')

print(f"AED-ar totalt:                  {len(aed_gdf)}")
print(f"AED-ar innanfor Kristiansand:    {len(aed_in_kommune)}")
print(f"AED-ar utanfor kommunegrensa:    {len(aed_gdf) - len(aed_in_kommune)}")

# Vis resultatet
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Alle AED-ar
kommune_gdf.plot(ax=axes[0], facecolor='#e8e8e8', edgecolor='#333', linewidth=1.5)
aed_gdf.plot(ax=axes[0], color='#3388ff', markersize=10, alpha=0.6)
axes[0].set_title(f'Alle AED-ar ({len(aed_gdf)})', fontsize=12)

# Berre innanfor kommune
kommune_gdf.plot(ax=axes[1], facecolor='#e8e8e8', edgecolor='#333', linewidth=1.5)
aed_in_kommune.plot(ax=axes[1], color='#27ae60', markersize=10, alpha=0.6)
axes[1].set_title(f'AED-ar innanfor Kristiansand ({len(aed_in_kommune)})', fontsize=12)

for ax in axes:
    ax.set_xlabel('Lengdegrad')
    ax.set_ylabel('Breddgrad')

plt.suptitle('Romleg filtrering — AED-ar innanfor kommunegrensa', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 3.3 Interaktivt kart med Folium

Vi lagar eit interaktivt kart med Folium som viser alle beredskapsressursar og AED-ar.
Kartet har MarkerCluster-gruppering og fargekoding etter type ressurs.

In [ ]:
# Interaktivt kart med Folium
m = folium.Map(location=[58.1414, 8.0842], zoom_start=12, tiles='OpenStreetMap')

# AED MarkerCluster
aed_cluster = MarkerCluster(name='AED Hjertestartarar').add_to(m)
for _, row in aed_gdf.iterrows():
    color = 'green' if row['is_open'] else 'red'
    status = 'Åpen' if row['is_open'] else 'Stengt'
    folium.CircleMarker(
        location=[row.latitude, row.longitude],
        radius=5, color=color, fill=True, fill_color=color, fill_opacity=0.7,
        popup=f"<b>{row['name']}</b><br>{row['address']}<br>Status: {status}",
        tooltip=f"{row['name']} ({status})"
    ).add_to(aed_cluster)

# Beredskapsressursar
beredskap_group = folium.FeatureGroup(name='Beredskapsressursar').add_to(m)
# Filtrer berre punkt-geometri for markørar
beredskap_points = beredskap_gdf[beredskap_gdf.geometry.geom_type == 'Point']
for _, row in beredskap_points.iterrows():
    folium.Marker(
        location=[row.geometry.y, row.geometry.x],
        popup=f"<b>{row['name']}</b><br>{row.get('category', '')}<br>{row.get('description', '')}",
        tooltip=row['name'],
        icon=folium.Icon(color='blue', icon='info-sign')
    ).add_to(beredskap_group)

# Kommunegrense
folium.GeoJson(
    kommune_gdf, name='Kommunegrense',
    style_function=lambda x: {'fillColor': 'transparent', 'color': '#333', 'weight': 2}
).add_to(m)

folium.LayerControl().add_to(m)
m

## 4. DuckDB — SQL-analyse av beredskapsdata

Vi brukar DuckDB for å køyre SQL-spørjingar direkte mot Pandas DataFrames.
DuckDB støttar romleg SQL og er nyttig for aggregering og analyse
utan å måtte sende data til ein ekstern database.

In [ ]:
# DuckDB: SQL-analyse av AED-data
con = duckdb.connect()

# Registrer AED-data som tabell
con.register('aed_data', aed_df)

# Spørjing 1: Antal AED per postområde
print("=" * 50)
print("SQL: Antal AED-ar per postområde")
print("=" * 50)
result1 = con.execute("""
    SELECT 
        post_code,
        COUNT(*) as antal_aed,
        SUM(CASE WHEN is_open THEN 1 ELSE 0 END) as antal_opne,
        ROUND(AVG(CASE WHEN is_open THEN 1.0 ELSE 0.0 END) * 100, 1) as prosent_opne
    FROM aed_data
    GROUP BY post_code
    ORDER BY antal_aed DESC
    LIMIT 15
""").fetchdf()
print(result1.to_string(index=False))

print("\n" + "=" * 50)
print("SQL: Statistikk over AED-dekning")
print("=" * 50)
result2 = con.execute("""
    SELECT 
        COUNT(*) as total_aed,
        SUM(CASE WHEN is_open THEN 1 ELSE 0 END) as opne,
        SUM(CASE WHEN NOT is_open THEN 1 ELSE 0 END) as stengte,
        ROUND(AVG(latitude), 4) as snitt_lat,
        ROUND(AVG(longitude), 4) as snitt_lng,
        ROUND(MIN(latitude), 4) as min_lat,
        ROUND(MAX(latitude), 4) as max_lat
    FROM aed_data
""").fetchdf()
print(result2.to_string(index=False))

con.close()

## 5. Vektoranalyse

### 5.1 Buffer: 500m beredskapssone rundt AED-ar

Vi opprettar ein buffersone på 500 meter rundt kvar AED.
Tanken er at ein AED bør vera tilgjengeleg innanfor 500m for
å vere nyttig i ein nødsituasjon (ca. 5-6 minutt gangavstand).

**Viktig**: Vi projiserer til UTM sone 32N (EPSG:25832) for korrekte avstandar i meter,
sidan EPSG:4326 brukar grader og gir feil bufferavstandar.

In [ ]:
# Projiser til UTM 32N for korrekte avstandar i meter
aed_utm = aed_gdf.to_crs(epsg=25832)
kommune_utm = kommune_gdf.to_crs(epsg=25832)

# Buffer: 500 meter rundt kvar AED
BUFFER_RADIUS = 500  # meter
aed_buffer = aed_utm.copy()
aed_buffer['geometry'] = aed_utm.geometry.buffer(BUFFER_RADIUS)

# Berre åpne AED-ar for beredskapsvurdering
aed_open_utm = aed_utm[aed_utm['is_open'] == True].copy()
aed_open_buffer = aed_open_utm.copy()
aed_open_buffer['geometry'] = aed_open_utm.geometry.buffer(BUFFER_RADIUS)

# Total dekningsareal (union av alle buffrar)
total_coverage = unary_union(aed_open_buffer.geometry)
coverage_area_km2 = total_coverage.area / 1e6

print(f"Bufferradius:        {BUFFER_RADIUS} meter")
print(f"Antal åpne AED-ar:   {len(aed_open_buffer)}")
print(f"Total dekningsareal: {coverage_area_km2:.2f} km²")

# Visualisering
fig, ax = plt.subplots(1, 1, figsize=(12, 8))
kommune_utm.plot(ax=ax, facecolor='#f5f5f5', edgecolor='#333', linewidth=1.5)
aed_open_buffer.plot(ax=ax, facecolor='#27ae60', edgecolor='none', alpha=0.3)
aed_open_utm.plot(ax=ax, color='#27ae60', markersize=8, alpha=0.8)

ax.set_title(f'Buffersone 500m rundt {len(aed_open_utm)} åpne AED-ar\nTotal dekking: {coverage_area_km2:.2f} km²',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Easting (m)')
ax.set_ylabel('Northing (m)')
plt.tight_layout()
plt.show()

print("\nDe grøne områda viser kor ein person er innanfor 500m gangavstand frå ein åpen AED.")
print("Kvite område innanfor kommunegrensa er ikkje dekka — her bør ein vurdere nye AED-ar.")

### 5.2 Overlay: Intersection — AED-dekking innanfor kommunen

Vi utfører ein **intersection** (overlay) mellom buffersona rundt AED-ane
og kommunegrensene. Dette gir oss nøyaktig kor mykje av kommunen som
er dekka av ein AED innanfor 500m.

Vi bereknar prosentvis dekking av kommunearealet — nyttig for
beredskapsvurdering i Totalforsvaret-kontekst.

In [ ]:
# Overlay: intersection mellom AED-buffrar og kommunegrense
# Lag ein GeoDataFrame av total coverage
coverage_gdf = gpd.GeoDataFrame(
    [{'id': 1, 'type': 'AED_coverage_500m', 'geometry': total_coverage}],
    crs='EPSG:25832'
)

# Intersection
overlay_result = gpd.overlay(coverage_gdf, kommune_utm, how='intersection')

# Berekn areal
kommune_area_km2 = kommune_utm.geometry.area.sum() / 1e6
covered_area_km2 = overlay_result.geometry.area.sum() / 1e6
coverage_pct = (covered_area_km2 / kommune_area_km2) * 100

print(f"Kommuneareal:         {kommune_area_km2:.2f} km²")
print(f"AED-dekka areal:      {covered_area_km2:.2f} km²")
print(f"Dekningsgrad:         {coverage_pct:.1f}%")
print(f"Ikkje dekka:          {kommune_area_km2 - covered_area_km2:.2f} km² ({100 - coverage_pct:.1f}%)")

# Visualisering
fig, ax = plt.subplots(1, 1, figsize=(12, 8))

# Kommune bakgrunn (ikkje-dekka = raud)
kommune_utm.plot(ax=ax, facecolor='#ffcccc', edgecolor='#333', linewidth=1.5, label='Ikkje dekka')

# Dekka areal (grønt)
overlay_result.plot(ax=ax, facecolor='#27ae60', edgecolor='none', alpha=0.5, label='AED-dekka (500m)')

# AED-punkt
aed_open_utm.plot(ax=ax, color='#1a6b35', markersize=5, alpha=0.7)

ax.set_title(
    f'Overlay: AED-dekking (500m buffer) innanfor Kristiansand kommune\n'
    f'Dekningsgrad: {coverage_pct:.1f}% av kommunearealet',
    fontsize=13, fontweight='bold'
)
ax.legend(loc='upper right', fontsize=10)
ax.set_xlabel('Easting (m)')
ax.set_ylabel('Northing (m)')
plt.tight_layout()
plt.show()

print("\nRaudt = område utan AED-dekning innanfor 500m.")
print("Grønt = område med minst éin AED innanfor 500m gangavstand.")
print("\nBeredskapsmessig bør dekningsgraden aukast i tettbygde strøk.")

### 5.3 Difference — Område utan AED-dekning

Vi brukar ein **difference**-overlay for å finne dei delane av kommunen
som ikkje har AED-dekning innanfor 500 meter. Desse områda representerer
potensielle «blindsoner» i beredskapen.

In [ ]:
# Overlay: difference — Kristiansand minus AED-buffrar
no_coverage = gpd.overlay(kommune_utm, coverage_gdf, how='difference')

no_coverage_km2 = no_coverage.geometry.area.sum() / 1e6

print(f"Areal utan AED-dekning (500m): {no_coverage_km2:.2f} km²")
print(f"Del av kommunearealet:         {(no_coverage_km2/kommune_area_km2)*100:.1f}%")

fig, ax = plt.subplots(1, 1, figsize=(12, 8))
no_coverage.plot(ax=ax, facecolor='#e94560', edgecolor='#333', linewidth=0.5, alpha=0.6)
aed_open_utm.plot(ax=ax, color='#27ae60', markersize=8, alpha=0.8)

ax.set_title(f'Områder utan AED-dekning (500m)\n{no_coverage_km2:.2f} km² utan dekning',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Easting (m)')
ax.set_ylabel('Northing (m)')
plt.tight_layout()
plt.show()

print("\nDe raude områda bør prioriterast for plassering av nye AED-ar.")

## 6. Romleg aggregering

### 6.1 Teljing av AED-ar per postområde

Vi utfører romleg aggregering ved å telje kor mange AED-ar som finst
i kvart postområde. Vi brukar `dissolve()` og `sjoin()` for å gruppere
punkt (AED-ar) innanfor polygon (postområde/grider).

Sidan vi ikkje har eksakte postnummergrenser som polygoner, lagar vi
eit rutenett (grid) over kommunen og tel AED-ar per rute.

In [ ]:
# Lag eit rutenett (1km x 1km grid) over kommunen
from shapely.geometry import box

bounds = kommune_utm.total_bounds
grid_size = 1000  # 1 km

xmin, ymin, xmax, ymax = bounds
grid_cells = []
cell_id = 0
for x in np.arange(xmin, xmax, grid_size):
    for y in np.arange(ymin, ymax, grid_size):
        cell = box(x, y, x + grid_size, y + grid_size)
        grid_cells.append({'cell_id': cell_id, 'geometry': cell})
        cell_id += 1

grid_gdf = gpd.GeoDataFrame(grid_cells, crs='EPSG:25832')

# Klipp grid til kommunegrensa
grid_clipped = gpd.overlay(grid_gdf, kommune_utm, how='intersection')

# Romleg join: knytt AED-punkt til rutenett
aed_in_grid = gpd.sjoin(aed_utm, grid_clipped, how='inner', predicate='within')

# Aggreger: tell AED-ar per rute
aed_count_per_cell = aed_in_grid.groupby('cell_id').size().reset_index(name='antal_aed')

# Join tilbake til grid
grid_with_count = grid_clipped.merge(aed_count_per_cell, on='cell_id', how='left')
grid_with_count['antal_aed'] = grid_with_count['antal_aed'].fillna(0).astype(int)

print(f"Rutenett: {len(grid_clipped)} celler (1km x 1km)")
print(f"Celler med AED:  {(grid_with_count['antal_aed'] > 0).sum()}")
print(f"Celler utan AED: {(grid_with_count['antal_aed'] == 0).sum()}")
print(f"\nFordelingstatistikk:")
print(grid_with_count['antal_aed'].describe())

# Visualisering: Choropleth-kart over AED-tettleik
fig, ax = plt.subplots(1, 1, figsize=(12, 8))
grid_with_count.plot(
    ax=ax, column='antal_aed', cmap='YlOrRd', edgecolor='#ccc', linewidth=0.3,
    legend=True, legend_kwds={'label': 'Antal AED per km²', 'shrink': 0.6}
)
kommune_utm.boundary.plot(ax=ax, edgecolor='#333', linewidth=1.5)
aed_utm.plot(ax=ax, color='black', markersize=3, alpha=0.4)

ax.set_title('Romleg aggregering: AED-tettleik per km²-rute i Kristiansand',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Easting (m)')
ax.set_ylabel('Northing (m)')
plt.tight_layout()
plt.show()

print("\nMørkare felt = fleire AED-ar = betre beredskap.")
print("Lyse/kvite felt bør evaluerast for beredskapsforbetringar.")

### 6.2 Aggregering med Pandas: AED-ar per postnummer

Vi brukar Pandas Groupby for å aggregere AED-data per postnummer.
Dette gir ein tabell over beredskapsnivå per område.

In [ ]:
# Aggregering med Pandas — AED per postnummer
agg = aed_df.groupby('post_code').agg(
    antal_aed=('asset_id', 'count'),
    antal_opne=('is_open', 'sum'),
    snitt_lat=('latitude', 'mean'),
    snitt_lng=('longitude', 'mean')
).reset_index()

agg['prosent_opne'] = (agg['antal_opne'] / agg['antal_aed'] * 100).round(1)
agg = agg.sort_values('antal_aed', ascending=False)

print("AED-ar per postnummer (topp 15):")
print(agg.head(15).to_string(index=False))

# Søylediagram
top_10 = agg.head(10)
fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(top_10['post_code'].astype(str), top_10['antal_aed'], color='#0f3460')
ax.set_title('Top 10 postnummer etter AED-antal', fontsize=13, fontweight='bold')
ax.set_xlabel('Postnummer')
ax.set_ylabel('Antal AED-ar')
ax.tick_params(axis='x', rotation=45)

# Merk søyler med antal
for bar, val in zip(bars, top_10['antal_aed']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            str(val), ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

### 6.3 Næraste beredskapsressurs frå kvar AED

Vi bereknar avstanden frå kvar AED til nærmaste beredskapsressurs
(sjukehus, legevakt osv.) for å vurdere kor godt integrert
hjartestarternettverket er med resten av beredskapsinfrastrukturen.

In [ ]:
# Berekn avstand frå kvar AED til nærmaste beredskapsressurs
from shapely.ops import nearest_points

beredskap_utm = beredskap_gdf.to_crs(epsg=25832)
beredskap_points_utm = beredskap_utm[beredskap_utm.geometry.geom_type == 'Point']

# Lag ein MultiPoint av alle beredskapsressursar
from shapely.geometry import MultiPoint
beredskap_multi = MultiPoint(beredskap_points_utm.geometry.tolist())

nearest_distances = []
for _, aed_row in aed_utm.iterrows():
    nearest_pt = nearest_points(aed_row.geometry, beredskap_multi)[1]
    dist_m = aed_row.geometry.distance(nearest_pt)
    nearest_distances.append(dist_m)

aed_utm_copy = aed_utm.copy()
aed_utm_copy['dist_beredskap_m'] = nearest_distances

print("Avstand frå AED til nærmaste beredskapsressurs (meter):")
print(aed_utm_copy['dist_beredskap_m'].describe().round(0))

# Histogram
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(aed_utm_copy['dist_beredskap_m'], bins=30, color='#0f3460', edgecolor='white', alpha=0.8)
ax.axvline(aed_utm_copy['dist_beredskap_m'].median(), color='#e94560', linestyle='--', linewidth=2,
           label=f'Median: {aed_utm_copy["dist_beredskap_m"].median():.0f}m')
ax.set_title('Avstand frå kvar AED til nærmaste beredskapsressurs', fontsize=13, fontweight='bold')
ax.set_xlabel('Avstand (meter)')
ax.set_ylabel('Antal AED-ar')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

## 7. Rasteranalyse

### 7.1 Last ned høgdedata (DEM)

Vi lastar ned ein Digital Elevation Model (DEM) for Kristiansand-området.
Vi brukar open Copernicus DEM (30m oppløysing) eller genererer ein syntetisk DEM
for demonstrasjon om nedlasting ikkje er tilgjengeleg.

DEM-en vert brukt til å berekne helningskart (slope), hillshade og identifisere
bratte område som kan vere utfordrande for evakuering.

In [ ]:
import rasterio
from rasterio.transform import from_bounds

# Definer område — Kristiansand sentrum
dem_bounds = {
    'west': 7.95,
    'east': 8.15,
    'south': 58.10,
    'north': 58.20
}

dem_path = 'kristiansand_dem.tif'

# Forsøk å laste SRTM/Copernicus DEM via OpenTopography eller liknande
try:
    # Prøv å hente frå OpenTopography API (krev nøkkel) eller anna kjelde
    dem_url = f"https://portal.opentopography.org/API/globaldem?demtype=COP30&south={dem_bounds['south']}&north={dem_bounds['north']}&west={dem_bounds['west']}&east={dem_bounds['east']}&outputFormat=GTiff"
    resp = requests.get(dem_url, timeout=60)
    if resp.status_code == 200 and len(resp.content) > 1000:
        with open(dem_path, 'wb') as f:
            f.write(resp.content)
        print(f"✓ DEM nedlasta: {dem_path} ({len(resp.content)} bytes)")
    else:
        raise Exception("API-svar ikkje gyldig")
except Exception as e:
    print(f"⚠ Kunne ikkje laste ned DEM: {e}")
    print("Genererer syntetisk DEM for demonstrasjon...")
    
    # Lag ein syntetisk DEM basert på Kristiansand-topografi
    # (kystby med ås-terreng nordover)
    nrows, ncols = 300, 400
    x = np.linspace(0, 10, ncols)
    y = np.linspace(0, 10, nrows)
    X, Y = np.meshgrid(x, y)
    
    # Simulert topografi: flatare mot sør (kyst), brattare nordover med ås
    Z = (
        50 * Y / 10 +                          # Generell stigning nordover
        30 * np.sin(X * 0.8) * np.cos(Y * 0.6) +  # Ås-rygg
        15 * np.sin(X * 2) * np.cos(Y * 1.5) +    # Småkupert terreng
        5 * np.random.randn(nrows, ncols)           # Mikroterreng
    )
    Z = np.clip(Z, 0, 200).astype(np.float32)
    
    transform = from_bounds(
        dem_bounds['west'], dem_bounds['south'],
        dem_bounds['east'], dem_bounds['north'],
        ncols, nrows
    )
    
    with rasterio.open(
        dem_path, 'w', driver='GTiff',
        height=nrows, width=ncols, count=1,
        dtype='float32', crs='EPSG:4326',
        transform=transform
    ) as dst:
        dst.write(Z, 1)
    
    print(f"✓ Syntetisk DEM generert: {dem_path} ({nrows}x{ncols} pikslar)")

# Les og vis DEM
with rasterio.open(dem_path) as src:
    dem_data = src.read(1)
    dem_transform = src.transform
    dem_crs = src.crs

fig, ax = plt.subplots(figsize=(12, 8))
im = ax.imshow(dem_data, cmap='terrain', extent=[
    dem_bounds['west'], dem_bounds['east'],
    dem_bounds['south'], dem_bounds['north']
])
plt.colorbar(im, ax=ax, label='Høgde (m.o.h.)', shrink=0.7)
ax.set_title('Digital terrengmodell (DEM) — Kristiansand', fontsize=13, fontweight='bold')
ax.set_xlabel('Lengdegrad')
ax.set_ylabel('Breddgrad')
plt.tight_layout()
plt.show()

print(f"DEM statistikk:")
print(f"  Min høgde:  {dem_data.min():.1f} m")
print(f"  Max høgde:  {dem_data.max():.1f} m")
print(f"  Snitt:      {dem_data.mean():.1f} m")
print(f"  Oppløysing: {dem_data.shape}")

### 7.2 Helningskart (Slope)

Vi bereknar helningskartet frå DEM-en ved hjelp av GDAL.
Helning i grader viser kor bratt terrenget er — område over 30° er
vanskelege å ta seg fram i til fots og kan vere problematiske
for evakuering og beredskapsinnsats.

#### CLI-kommando med GDAL:
```bash
gdaldem slope kristiansand_dem.tif kristiansand_slope.tif -of GTiff -b 1 -s 1.0
```

In [ ]:
# Berekn helning (slope) frå DEM
# Alternativ 1: Bruk GDAL via subprocess
import subprocess

slope_path = 'kristiansand_slope.tif'

try:
    # Prøv GDAL CLI
    result = subprocess.run(
        ['gdaldem', 'slope', dem_path, slope_path, '-of', 'GTiff', '-b', '1'],
        capture_output=True, text=True, timeout=30
    )
    if result.returncode == 0:
        print("✓ Helningskart generert med GDAL")
    else:
        raise Exception(result.stderr)
except Exception as e:
    print(f"⚠ GDAL ikkje tilgjengeleg ({e})")
    print("Bereknar helning manuelt med NumPy...")
    
    # Manuell slope-berekning med gradient
    # Pikselstorleik i meter (tilnæring ved ~58°N)
    pixel_size_x = abs(dem_transform[0]) * 111320 * np.cos(np.radians(58.15))  # m
    pixel_size_y = abs(dem_transform[4]) * 110540  # m
    
    dy, dx = np.gradient(dem_data, pixel_size_y, pixel_size_x)
    slope_rad = np.arctan(np.sqrt(dx**2 + dy**2))
    slope_deg = np.degrees(slope_rad).astype(np.float32)
    
    with rasterio.open(
        slope_path, 'w', driver='GTiff',
        height=slope_deg.shape[0], width=slope_deg.shape[1],
        count=1, dtype='float32', crs='EPSG:4326',
        transform=dem_transform
    ) as dst:
        dst.write(slope_deg, 1)
    print(f"✓ Helningskart generert manuelt: {slope_path}")

# Les og vis helningskart
with rasterio.open(slope_path) as src:
    slope_data = src.read(1)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Helningskart
im1 = axes[0].imshow(slope_data, cmap='RdYlGn_r', vmin=0, vmax=45, extent=[
    dem_bounds['west'], dem_bounds['east'], dem_bounds['south'], dem_bounds['north']
])
plt.colorbar(im1, ax=axes[0], label='Helning (grader)', shrink=0.7)
axes[0].set_title('Helningskart', fontsize=13, fontweight='bold')

# Brattare enn 30°
steep = np.where(slope_data > 30, slope_data, np.nan)
im2 = axes[1].imshow(dem_data, cmap='terrain', alpha=0.3, extent=[
    dem_bounds['west'], dem_bounds['east'], dem_bounds['south'], dem_bounds['north']
])
axes[1].imshow(steep, cmap='Reds', alpha=0.8, extent=[
    dem_bounds['west'], dem_bounds['east'], dem_bounds['south'], dem_bounds['north']
])
axes[1].set_title('Område brattare enn 30° (raudt)', fontsize=13, fontweight='bold')

for ax in axes:
    ax.set_xlabel('Lengdegrad')
    ax.set_ylabel('Breddgrad')

plt.tight_layout()
plt.show()

steep_pct = (slope_data > 30).sum() / slope_data.size * 100
print(f"\nHelningsstatistikk:")
print(f"  Min helning:         {slope_data.min():.1f}°")
print(f"  Max helning:         {slope_data.max():.1f}°")
print(f"  Snitt helning:       {slope_data.mean():.1f}°")
print(f"  Pikslar > 30°:       {(slope_data > 30).sum()} ({steep_pct:.1f}%)")
print(f"\nBratte område er vanskelege for evakuering og beredskapsutstyr.")

### 7.3 Vektorisering av bratte område (Polygonize)

Vi konverterer helningsdata > 30° (raster) til vektordata (polygon).
Dette gjer det mogleg å bruke dei bratte områda i vektoranalyse,
t.d. overlay med beredskapsressursar.

#### CLI-kommando med GDAL:
```bash
gdal_calc.py -A kristiansand_slope.tif --outfile=steep_mask.tif --calc="A>30" --NoDataValue=0
gdal_polygonize.py steep_mask.tif steep_areas.gpkg steep_areas
```

In [ ]:
from rasterio.features import shapes
from shapely.geometry import shape

# Lag binære data: 1 = brattare enn 30°, 0 = elles
steep_mask = (slope_data > 30).astype(np.uint8)

# Polygonize
with rasterio.open(slope_path) as src:
    transform = src.transform

steep_polygons = []
for geom, value in shapes(steep_mask, mask=(steep_mask == 1), transform=transform):
    steep_polygons.append({
        'geometry': shape(geom),
        'steep': True
    })

if steep_polygons:
    steep_gdf = gpd.GeoDataFrame(steep_polygons, crs='EPSG:4326')
    steep_gdf['area_m2'] = steep_gdf.to_crs(epsg=25832).geometry.area
    
    print(f"✓ {len(steep_gdf)} polygon(ar) med helning > 30°")
    print(f"Totalt areal (bratt): {steep_gdf['area_m2'].sum()/1e6:.3f} km²")
    
    # Vis på kart
    fig, ax = plt.subplots(figsize=(12, 8))
    steep_gdf.plot(ax=ax, facecolor='#e94560', edgecolor='#c0392b', linewidth=0.5, alpha=0.6)
    ax.set_title(f'Vektoriserte bratte område (> 30°) — {len(steep_gdf)} polygon',
                 fontsize=13, fontweight='bold')
    ax.set_xlabel('Lengdegrad')
    ax.set_ylabel('Breddgrad')
    plt.tight_layout()
    plt.show()
    
    # Lagre som GeoPackage
    steep_gdf.to_file('steep_areas.gpkg', driver='GPKG')
    print("✓ Lagra som steep_areas.gpkg (GeoPackage)")
else:
    print("Ingen område over 30° funne i DEM-en.")
    steep_gdf = gpd.GeoDataFrame(columns=['geometry', 'steep'], crs='EPSG:4326')

### 7.4 Hillshade — skyggelegging av terrenget

Vi lagar to ulike hillshade-kart ved å variere parametrane for
lyskjelde (azimuth og altitude). Hillshade gir ein visuell
framstilling av terrenget som er lettare å tolke enn rå høgdedata.

#### CLI-kommandoar med GDAL:
```bash
# Hillshade 1: Standard (sol frå nordvest, lav vinkel)
gdaldem hillshade kristiansand_dem.tif hillshade_nw.tif -az 315 -alt 35

# Hillshade 2: Sol frå aust, høgare vinkel
gdaldem hillshade kristiansand_dem.tif hillshade_east.tif -az 90 -alt 60
```

In [ ]:
def compute_hillshade(elevation, azimuth=315, altitude=35, transform=None):
    """
    Berekn hillshade frå DEM.
    azimuth: retning til ljoset (grader frå nord)
    altitude: vinkel til ljoset over horisonten (grader)
    """
    # Pikselstorleik i meter
    if transform:
        pixel_x = abs(transform[0]) * 111320 * np.cos(np.radians(58.15))
        pixel_y = abs(transform[4]) * 110540
    else:
        pixel_x = pixel_y = 30.0
    
    dy, dx = np.gradient(elevation, pixel_y, pixel_x)
    slope = np.arctan(np.sqrt(dx**2 + dy**2))
    aspect = np.arctan2(-dx, dy)
    
    az_rad = np.radians(azimuth)
    alt_rad = np.radians(altitude)
    
    hillshade = (
        np.sin(alt_rad) * np.cos(slope) +
        np.cos(alt_rad) * np.sin(slope) * np.cos(az_rad - aspect)
    )
    
    return np.clip(hillshade * 255, 0, 255).astype(np.uint8)

# Hillshade 1: Sol frå nordvest (315°), lav vinkel (35°)
hs1 = compute_hillshade(dem_data, azimuth=315, altitude=35, transform=dem_transform)

# Hillshade 2: Sol frå aust (90°), høg vinkel (60°)
hs2 = compute_hillshade(dem_data, azimuth=90, altitude=60, transform=dem_transform)

# Lagre som GeoTIFF
for hs_data, hs_name, hs_file in [
    (hs1, 'Hillshade NV (az=315°, alt=35°)', 'hillshade_nw.tif'),
    (hs2, 'Hillshade Aust (az=90°, alt=60°)', 'hillshade_east.tif')
]:
    with rasterio.open(
        hs_file, 'w', driver='GTiff',
        height=hs_data.shape[0], width=hs_data.shape[1],
        count=1, dtype='uint8', crs='EPSG:4326',
        transform=dem_transform
    ) as dst:
        dst.write(hs_data, 1)
    print(f"✓ Lagra {hs_file}")

# Visualisering
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

extent = [dem_bounds['west'], dem_bounds['east'], dem_bounds['south'], dem_bounds['north']]

axes[0].imshow(hs1, cmap='gray', extent=extent)
axes[0].set_title('Hillshade 1: Sol frå NV (az=315°, alt=35°)', fontsize=12, fontweight='bold')

axes[1].imshow(hs2, cmap='gray', extent=extent)
axes[1].set_title('Hillshade 2: Sol frå aust (az=90°, alt=60°)', fontsize=12, fontweight='bold')

for ax in axes:
    ax.set_xlabel('Lengdegrad')
    ax.set_ylabel('Breddgrad')

plt.suptitle('Hillshade med ulike lysparameter', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nHillshade 1 (NV, lav vinkel) framhevar detaljar i terrenget.")
print("Hillshade 2 (aust, høg vinkel) gir ein mykare, meir generell skygging.")
print("Ulik lysretning avdekkar ulike terrengeigenskapar.")

## 8. Kombinert analyse: Bratte område og beredskap

Vi kombinerer resultata frå raster- og vektoranalysen for å identifisere
AED-ar som ligg i bratte område. Desse kan vere vanskelegare å nå
i ein nødsituasjon.

In [ ]:
# Kombinert analyse: AED-ar i bratte område
if len(steep_gdf) > 0:
    aed_in_steep = gpd.sjoin(aed_gdf, steep_gdf, how='inner', predicate='within')
    print(f"AED-ar i bratte område (>30°): {len(aed_in_steep)}")
    print(f"AED-ar i normalt terreng:      {len(aed_gdf) - len(aed_in_steep)}")
    
    if len(aed_in_steep) > 0:
        print("\nDesse AED-ane kan vere vanskelegare å nå:")
        print(aed_in_steep[['name', 'address', 'is_open']].head(10))
else:
    print("Ingen bratte område funne — alle AED-ar er i normalt terreng.")
    print("(Dette kan skuldast den syntetiske DEM-en.)")

# Samandrag
print("\n" + "=" * 60)
print("SAMANDRAG — Beredskapsanalyse Kristiansand")
print("=" * 60)
print(f"Totalt AED-ar:          {len(aed_gdf)}")
print(f"Åpne AED-ar:            {aed_gdf['is_open'].sum()}")
print(f"AED i kommunen:         {len(aed_in_kommune)}")
print(f"AED-dekking (500m):     {coverage_pct:.1f}% av kommunearealet")
print(f"Beredskapsressursar:    {len(beredskap_gdf)}")
print(f"Bratte områder (>30°):  {steep_pct:.1f}% av DEM-arealet")
print(f"\nKonklusjon: Beredskapsdekninga i sentrale Kristiansand er")
print(f"rimeleg god, men det er områder i utkanten som treng fleire AED-ar.")

## 9. Oppsummering

### Kva vi har gjort:

1. **Data Ingest**: Henta tre datasett — lokal GeoJSON, API (Hjertestarterregister), og WFS (GeoNorge)
2. **Filtrering**: Attributtfiltrering (åpen/stengt) og romleg filtrering (innanfor kommune)
3. **Visualisering**: Statisk (matplotlib/geopandas.plot) og interaktiv (Folium)
4. **DuckDB**: SQL-analyse av AED-fordeling
5. **Buffer**: 500m beredskapssone rundt AED-ar
6. **Overlay**: Intersection og difference mot kommunegrense
7. **Aggregering**: AED-tettleik per km²-rute og per postnummer
8. **Raster**: DEM, helningskart, vektorisering, to hillshade-kart

### Verktøy brukt:
- Pandas, GeoPandas, DuckDB, Folium, Matplotlib, Rasterio, Shapely, NumPy
- GDAL (CLI-kommandoar dokumentert i markdown)

### Relevans for Totalforsvaret:
Analysane viser korleis romleg data kan brukast for å vurdere og forbetre
beredskapsdekninga i ein kommune. AED-dekningsanalysen identifiserer
blindsoner der nye hjertestartarar bør plasserast, medan terrenganalysen
viser område som er vanskelegare tilgjengelege for beredskapsinnsats.